# 02 — Face Recognition Setup

Builds `app/models/face_db.pkl`: a dict of `{customer_id: [128-d encodings]}` used by `cv_service.recognize_face()` for returning-customer detection.

**Ethics note:** use a demo dataset of *consenting* sample faces (e.g. an LFW subset) or your own enrolled photos — never scrape faces without consent. Discuss consent, privacy, and bias in your final report.

In [ ]:
!pip install face_recognition --quiet  # requires dlib; see repo README/Dockerfile for build notes

In [ ]:
import os
import pickle
import face_recognition

print('face_recognition ready')

## 1. Config
Expects a folder per customer:
```
data/faces/
  customer_001/
    photo1.jpg
    photo2.jpg
  customer_002/
    photo1.jpg
```
Multiple photos per customer improve match robustness.

In [ ]:
FACES_DIR = '../data/faces'   # <-- point this at your enrollment photos
OUT_PATH = '../app/models/face_db.pkl'

## 2. Build encodings for every enrolled customer

In [ ]:
face_db = {}

for customer_id in sorted(os.listdir(FACES_DIR)):
    customer_dir = os.path.join(FACES_DIR, customer_id)
    if not os.path.isdir(customer_dir):
        continue
    encodings = []
    for fname in os.listdir(customer_dir):
        path = os.path.join(customer_dir, fname)
        image = face_recognition.load_image_file(path)
        found = face_recognition.face_encodings(image)
        if found:
            encodings.append(found[0])
        else:
            print(f'No face found in {path}, skipping')
    if encodings:
        face_db[customer_id] = encodings
        print(f'{customer_id}: {len(encodings)} encoding(s)')

print(f'\nTotal enrolled customers: {len(face_db)}')

## 3. Sanity check: self-match test
Each customer's own first photo should match themselves with a small distance.

In [ ]:
import numpy as np

for customer_id, encs in face_db.items():
    if len(encs) < 2:
        continue
    dist = face_recognition.face_distance([encs[0]], encs[1])[0]
    print(f'{customer_id}: self-distance = {dist:.4f} (lower = more similar)')

## 4. Save the face DB
Saved to `app/models/face_db.pkl` — matches the path `cv_service.py` expects.

In [ ]:
os.makedirs('../app/models', exist_ok=True)
with open(OUT_PATH, 'wb') as f:
    pickle.dump(face_db, f)
print(f'Saved {len(face_db)} customers to {OUT_PATH}')